# Bias & Fairness Audit

**The premise.** A classifier that is accurate in aggregate can be systematically unfair to a protected group. A PM's job is to know whether this is happening, quantify it, and either mitigate or make the tradeoff explicit to stakeholders.

**What this notebook produces.**
1. A baseline classifier on a tabular dataset (UCI Adult), trained without any fairness intervention.
2. Two group-fairness metrics — **demographic parity difference** and **equalized odds difference** — computed across the protected attribute.
3. A reweighting mitigation: re-weight training samples so the protected groups have equal representation in each outcome.
4. A before/after comparison showing the fairness/accuracy tradeoff.

**Why not a fairness library.** `aif360`, `fairlearn`, and others exist and are production-appropriate. For a PM portfolio, a from-scratch implementation shows the PM understands what the library is doing rather than just calling it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load UCI Adult (income) dataset

The target is whether a person earns >$50k/year. The protected attribute for this audit is `sex`. This dataset is a known fairness benchmark because labor-market income is historically biased on sex — we expect to find that bias reflected in a naive model.

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True, parser="liac-arff")
df = adult.frame.dropna().copy()

y = (df["class"] == ">50K").astype(int).values
protected = (df["sex"] == "Male").astype(int).values  # 1 = Male, 0 = Female

feature_cols = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]
X = df[feature_cols].astype(float).values

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, protected, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train: {len(X_train_s)}  Test: {len(X_test_s)}")
print(f"Positive rate — Male: {y_train[g_train==1].mean():.3f}, Female: {y_train[g_train==0].mean():.3f}")

## 2. Baseline classifier and fairness metrics

**Demographic parity difference (DPD).** |P(Ŷ=1|A=0) − P(Ŷ=1|A=1)|. Zero means the model predicts positive at equal rates across groups. Large values mean the model's decisions are correlated with the protected attribute.

**Equalized odds difference (EOD).** Maximum absolute difference in true-positive rate and false-positive rate between groups. Zero means the model is equally accurate across groups.

In [ ]:
def fairness_metrics(y_true, y_pred, group):
    mask_a = group == 0
    mask_b = group == 1
    pred_rate_a = y_pred[mask_a].mean()
    pred_rate_b = y_pred[mask_b].mean()
    dpd = abs(pred_rate_a - pred_rate_b)

    def tpr(y_t, y_p):
        pos = y_t == 1
        return y_p[pos].mean() if pos.any() else 0.0
    def fpr(y_t, y_p):
        neg = y_t == 0
        return y_p[neg].mean() if neg.any() else 0.0

    tpr_diff = abs(tpr(y_true[mask_a], y_pred[mask_a]) - tpr(y_true[mask_b], y_pred[mask_b]))
    fpr_diff = abs(fpr(y_true[mask_a], y_pred[mask_a]) - fpr(y_true[mask_b], y_pred[mask_b]))
    eod = max(tpr_diff, fpr_diff)

    accuracy = (y_true == y_pred).mean()
    return {"accuracy": accuracy, "DPD": dpd, "EOD": eod,
            "pos_rate_female": pred_rate_a, "pos_rate_male": pred_rate_b}

baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train_s, y_train)
y_pred_baseline = baseline.predict(X_test_s)

metrics_baseline = fairness_metrics(y_test, y_pred_baseline, g_test)
print("Baseline (no mitigation):")
for k, v in metrics_baseline.items():
    print(f"  {k}: {v:.4f}")

## 3. Mitigation — sample reweighting

For each (group, label) cell, assign a weight inversely proportional to its observed frequency. This equalizes the joint distribution of (protected attribute, outcome) in training, removing the statistical signal that correlates group with label.

In [ ]:
def reweighting_weights(y, group):
    n = len(y)
    weights = np.ones(n)
    for g_val in np.unique(group):
        for y_val in np.unique(y):
            mask = (group == g_val) & (y == y_val)
            if mask.any():
                # expected freq under independence = P(A=g) * P(Y=y)
                expected = (group == g_val).mean() * (y == y_val).mean()
                observed = mask.mean()
                weights[mask] = expected / observed
    return weights

w_train = reweighting_weights(y_train, g_train)

mitigated = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
mitigated.fit(X_train_s, y_train, sample_weight=w_train)
y_pred_mitigated = mitigated.predict(X_test_s)

metrics_mitigated = fairness_metrics(y_test, y_pred_mitigated, g_test)
print("After reweighting mitigation:")
for k, v in metrics_mitigated.items():
    print(f"  {k}: {v:.4f}")

## 4. Before / after visualization

In [ ]:
comparison = pd.DataFrame({
    "Baseline": [metrics_baseline["accuracy"], metrics_baseline["DPD"], metrics_baseline["EOD"]],
    "Mitigated": [metrics_mitigated["accuracy"], metrics_mitigated["DPD"], metrics_mitigated["EOD"]],
}, index=["Accuracy", "Demographic Parity Diff", "Equalized Odds Diff"])

fig, ax = plt.subplots(figsize=(8, 5))
comparison.plot(kind="bar", ax=ax, color=["#bf3989", "#2ea043"])
ax.set_ylabel("Score")
ax.set_title("Fairness / accuracy tradeoff — reweighting mitigation")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=9)
plt.tight_layout()
plt.show()

print("\nTrade-off summary:")
print(f"  Accuracy change: {metrics_mitigated['accuracy'] - metrics_baseline['accuracy']:+.4f}")
print(f"  DPD reduction:   {metrics_baseline['DPD'] - metrics_mitigated['DPD']:+.4f}")
print(f"  EOD reduction:   {metrics_baseline['EOD'] - metrics_mitigated['EOD']:+.4f}")

## PM implications

| Outcome | What to do |
|---|---|
| **Accuracy dropped but fairness improved** | This is usually the correct trade when the feature affects consequential decisions (lending, hiring, medical triage). Frame the tradeoff in cost terms: 'we lose N correct predictions to prevent M unfair ones.' |
| **Fairness barely moved** | Reweighting only fixes representation bias. If labels themselves encode historical bias, you need a different intervention (label-side correction or counterfactual augmentation). |
| **Accuracy dropped meaningfully** | Investigate whether the unfairness was driven by a legitimate feature (e.g., hours-worked) or a proxy for the protected attribute. If legitimate, reweighting is the wrong tool. |

**What doesn't go in the exec deck:** the raw metric numbers. What does: a concrete decision table — 'at the mitigated threshold we approve X% of group A vs Y% of group B, versus A% vs B% before; the mitigated model reduces the gap by N pp at the cost of M correct decisions per 10k. Recommendation: accept / reject / escalate to legal.'

Fairness work is most useful when it changes what ships, not when it fills a compliance doc.